In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [17]:
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    probabilities = counts / np.sum(counts)
    entropy_value = -np.sum(probabilities * np.log2(probabilities))
    return entropy_value
def information_gain(data, feature, target="Class"):
    total_entropy = entropy(data[target])
    values, counts = np.unique(data[feature], return_counts=True)

    weighted_entropy = 0
    for i in range(len(values)):
        subset = data[data[feature] == values[i]][target]
        weight = counts[i] / np.sum(counts)
        weighted_entropy += weight * entropy(subset)

    return total_entropy - weighted_entropy
def id3(data, original_data, features, target="Class", parent_node_class=None):

    if len(np.unique(data[target])) == 1:
        return np.unique(data[target])[0]

    if len(data) == 0:
        return np.unique(original_data[target])[np.argmax(
            np.unique(original_data[target], return_counts=True)[1])]

    if len(features) == 0:
        return parent_node_class   

    parent_node_class = np.unique(data[target])[np.argmax(
        np.unique(data[target], return_counts=True)[1])]

    gains = [information_gain(data, feature, target) for feature in features]
    best_feature = features[np.argmax(gains)]

    tree = {best_feature: {}}

    remaining_features = [f for f in features if f != best_feature]

    for value in np.unique(data[best_feature]):
        subset = data[data[best_feature] == value]
        subtree = id3(subset, original_data, remaining_features, target, parent_node_class)
        tree[best_feature][value] = subtree

    return tree

data = pd.DataFrame({
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Mild', 'Cool'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak'],
    'Class': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No']
})
features = list(data.columns[:-1])
tree = id3(data, data, features)
print(tree)


{'Outlook': {'Overcast': 'Yes', 'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}}, 'Sunny': 'No'}}
